In [1]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [2]:
# extract text files 
def load_pdf_files(data):
    loader = DirectoryLoader(
        data , glob="*.pdf" , 
        loader_cls = PyPDFLoader 
    )
    documents = loader.load()
    return documents 

In [5]:
extract = load_pdf_files(r"C:\Users\ymant\OneDrive\Desktop\ML_Pro_26_Summer\Chatbot_Complete_Project\Medical-Chatbot\data")


In [6]:
len(extract)

4505

In [7]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs():
    """
    Given a list of document objects , return a new list of document objects containing only
    'source' in metadata and the original page_content
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get('source')
        minimal_docs.append(
            Document(
                page_content=doc.page_content , 
                metadata = {'source':src}
            )
        )
    return minimal_docs    

In [11]:
# ensure the function can access the loaded documents
docs = extract
minimal_docs = filter_to_minimal_docs()


## Chunking

In [14]:
#Splitting the whole document into small chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )

    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

 



In [15]:
text_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunk)}")


Number of chunks: 40000


## Embedding Model

In [16]:
import torch
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """

    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
        #model_kwargs={
            #"device": "cuda" if torch.cuda.is_available() else "cpu"
        #}
    )

    return embeddings

In [17]:
embedding = download_embeddings()

C:\Users\ymant\AppData\Local\Temp\ipykernel_31484\3352876708.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [20]:
vector = embedding.embed_query('Hello World')
len(vector)

384

LOADING API KEYS 

In [21]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [22]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [23]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key = pinecone_api_key)


In [24]:
pc

Creating a database using index 

In [27]:
from pinecone import ServerlessSpec

index_name = 'medical-chatbot'

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384, 
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1') 
    )


In [29]:
index = pc.Index(index_name)


## Storing the vector 

In [30]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunk , 
    embedding=embedding , 
    index_name = index_name

)

In [31]:
# Loading existing Index 
from langchain_pinecone import PineconeVectorStore
# embed each chunk and upsert the embeddigns into your pinecone
docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name , 
    embedding = embedding 
)

## Add more data to the existing Pinecone index 

In [32]:
myself = Document(
    page_content = "Hello , I am Yukteswar , a mathematics turned CS enthusiast" , 
    metadata = {'source':'Github'}
)

In [33]:
docsearch.add_documents(documents=[myself])

['960bc964-0d99-44dc-948e-bc4f8acc25b0']

## Creating the retriever

In [34]:
retriever = docsearch.as_retriever(search_type='similarity' , search_kwargs={'k':3})

In [35]:
retriever_docs = retriever.invoke("What is elephantiasis ?")
retriever_docs

[Document(id='1ede41f5-b252-48d4-8de3-a397f566698c', metadata={'source': 'C:\\Users\\ymant\\OneDrive\\Desktop\\ML_Pro_26_Summer\\Chatbot_Complete_Project\\Medical-Chatbot\\data\\Medicine.pdf'}, page_content='Elephantiasis\nDefinition\nThe word elephantiasis is a vivid and accurate\nterm for the syndrome it describes: the gross (visible)\nenlargement of the arms, legs, or genitals to elephan-\ntoid size.\nDescription\nTrue elephantiasis is the result of a parasitic infec-\ntion caused by three specific kinds of round worms.\nThe long, threadlike worms block the body’s lympha-\ntic system–a network of channels, lymph nodes, and\norgans that helps maintain proper fluid levels in the'),
 Document(id='18325bfe-b0a0-4e74-bbbe-525a41b06b2b', metadata={'source': 'C:\\Users\\ymant\\OneDrive\\Desktop\\ML_Pro_26_Summer\\Chatbot_Complete_Project\\Medical-Chatbot\\data\\Medicine.pdf'}, page_content='Man suffering from elephantiasis. (Photograph by C. James\nWebb, Phototake NYC. Reproduced by permis

Connecting The LLM

In [36]:
from langchain_openai import ChatOpenAI
chat_model = ChatOpenAI(model='gpt-4o')

In [39]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [40]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [41]:
question_answer_chain = create_stuff_documents_chain( chat_model ,prompt)
rag_chain = create_retrieval_chain(retriever , question_answer_chain)

In [42]:
response = rag_chain.invoke({"input" : "What is Acromegaly and gigantism ? "})
print(response['answer'])

Acromegaly is a disorder characterized by the abnormal release of a chemical from the pituitary gland in the brain, leading to increased growth in bone and soft tissue, along with other bodily disturbances. Gigantism is related to acromegaly and is also due to the overproduction of growth-related hormones by the pituitary gland. Both conditions are linked to either hyperactivity or other dysfunctions of the pituitary.
